In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
pd.set_option('mode.chained_assignment', None)

In [3]:
df = pd.read_parquet("../../novus/matchingnemo/scratch/ampnet_data/Portland/", engine = "pyarrow")

In [4]:
# safegraph data coverage (safegraph match / total FEMA buildings)
len(df[df['t'].notna()])/len(df['t'])

0.879825690962613

In [5]:
dat = df.dropna() # training set

In [6]:
dat[dat.duplicated(subset = ['build_id', 'date_range_start', 't'])].sort_values(by = ['date_range_start', 'build_id', 't'])

,build_id,occ_cls,prim_occ,sqmeters,sqfeet,censuscode,uuid,safegraph_place_id,date_range_start,t,visits
273923,12768,Industrial,Light,6288.146973,67684.984375,41067032001,{ab426d8b-4053-4095-82fd-9d76414326c7},sg:3c484805b84d4c25bffea700b8838c57,2018-12-31 08:00:00,0.0,0.0
273924,12768,Industrial,Light,6288.146973,67684.984375,41067032001,{ab426d8b-4053-4095-82fd-9d76414326c7},sg:3c484805b84d4c25bffea700b8838c57,2018-12-31 08:00:00,1.0,0.0
273925,12768,Industrial,Light,6288.146973,67684.984375,41067032001,{ab426d8b-4053-4095-82fd-9d76414326c7},sg:3c484805b84d4c25bffea700b8838c57,2018-12-31 08:00:00,2.0,0.0
273926,12768,Industrial,Light,6288.146973,67684.984375,41067032001,{ab426d8b-4053-4095-82fd-9d76414326c7},sg:3c484805b84d4c25bffea700b8838c57,2018-12-31 08:00:00,3.0,0.0
273927,12768,Industrial,Light,6288.146973,67684.984375,41067032001,{ab426d8b-4053-4095-82fd-9d76414326c7},sg:3c484805b84d4c25bffea700b8838c57,2018-12-31 08:00:00,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
89324579,8214514,Commercial,Retail Trade,20751.080078,223362.546875,41051007300,{9e5b5d1b-9ff4-47f0-beda-3e9f6c12b0e1},sg:7506ab9cd1c04c748c6c4a66e1555675,2019-12-23 08:00:00,167.0,0.0
89324747,8214514,Commercial,Retail Trade,20751.080078,223362.546875,41051007300,{9e5b5d1b-9ff4-47f0-beda-3e9f6c12b0e1},sg:82320a98639a4752b023e0298fffefea,2019-12-23 08:00:00,167.0,0.0
89324916,8214514,Commercial,Retail Trade,20751.080078,223362.546875,41051007300,{9e5b5d1b-9ff4-47f0-beda-3e9f6c12b0e1},sg:828fbb1cf6da47ccb556c5345d3b38f7,2019-12-23 08:00:00,167.0,0.0
89325084,8214514,Commercial,Retail Trade,20751.080078,223362.546875,41051007300,{9e5b5d1b-9ff4-47f0-beda-3e9f6c12b0e1},sg:813976f61c374ac9b02401b01250ea8c,2019-12-23 08:00:00,167.0,0.0


In [7]:
dat['sq_q'] = pd.qcut(dat['sqmeters'], q=4, labels=[1, 2, 3, 4])

In [8]:
temporal_mean = dat.groupby('t')['visits'].mean()

In [9]:
pivot = pd.pivot_table(dat.dropna(), values = 'visits', index = ['t', 'sq_q'], columns = 'prim_occ', aggfunc = 'mean', observed = True)
t_values = pivot.index.get_level_values('t')

# Convert Index → Series, preserving pivot's row index
fallback = pd.Series(
    t_values.map(temporal_mean),
    index=pivot.index
)

for col in pivot.columns:
    pivot[col] = pivot[col].fillna(fallback)

In [10]:
pivot.reset_index().head()

prim_occ,t,sq_q,Agriculture,Aviation,Colleges/Universities,Community Center,Emergency Response,Energy Control Monitoring,Entertainment and Recreation,General Services,...,Other Educational Buildings,Parking,Personal and Repair Services,Pre-K - 12 Schools,Professional/Technical Services,Religious,Retail Trade,Theaters,Unclassified,Wholesale Trade
0,0.0,1,0.00000,0.058030,0.053846,0.013921,0.025316,0.058030,0.088660,0.029297,...,0.036338,0.000000,0.037344,0.024427,0.027882,0.033224,0.052193,0.028846,0.038168,0.061856
1,0.0,2,0.05803,0.058030,0.074627,0.031496,0.041719,0.058030,0.121144,0.054245,...,0.032700,0.096154,0.039544,0.039645,0.026720,0.035507,0.037051,0.000000,0.034064,0.048077
2,0.0,3,0.05803,0.384615,0.088702,0.010657,0.048128,0.058030,0.055524,0.054348,...,0.054764,0.094203,0.027095,0.024674,0.037225,0.035178,0.038238,0.058030,0.050395,0.018868
3,0.0,4,0.05803,12.584135,0.082377,0.042453,0.063694,0.115385,0.073320,0.096678,...,0.030389,0.027723,0.036686,0.055365,0.055047,0.066374,0.064601,0.007812,0.062616,0.136364
4,1.0,1,0.00000,0.041771,0.026923,0.009281,0.040506,0.041771,0.061673,0.009766,...,0.016073,0.000000,0.027166,0.022901,0.021497,0.023026,0.041625,0.057692,0.026172,0.051546


In [11]:
dat[dat.duplicated(['build_id', 't'], keep=False)]

,build_id,occ_cls,prim_occ,sqmeters,sqfeet,censuscode,uuid,safegraph_place_id,date_range_start,t,visits,sq_q
157694,678266,Education,Pre-K - 12 Schools,872.624207,9392.839844,41051009903,{37403d88-1494-43d3-8947-9d8300825241},sg:ee72a81f6547452291503c55f94f270d,2018-12-31 08:00:00,0.0,0.0,2
157695,678266,Education,Pre-K - 12 Schools,872.624207,9392.839844,41051009903,{37403d88-1494-43d3-8947-9d8300825241},sg:ee72a81f6547452291503c55f94f270d,2018-12-31 08:00:00,1.0,0.0,2
157696,678266,Education,Pre-K - 12 Schools,872.624207,9392.839844,41051009903,{37403d88-1494-43d3-8947-9d8300825241},sg:ee72a81f6547452291503c55f94f270d,2018-12-31 08:00:00,2.0,0.0,2
157697,678266,Education,Pre-K - 12 Schools,872.624207,9392.839844,41051009903,{37403d88-1494-43d3-8947-9d8300825241},sg:ee72a81f6547452291503c55f94f270d,2018-12-31 08:00:00,3.0,0.0,2
157698,678266,Education,Pre-K - 12 Schools,872.624207,9392.839844,41051009903,{37403d88-1494-43d3-8947-9d8300825241},sg:ee72a81f6547452291503c55f94f270d,2018-12-31 08:00:00,4.0,0.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...
96756296,488064,Commercial,Retail Trade,466.449554,5020.816406,41051004103,{04b67782-0d55-45cc-8a67-9e565e7c3bf1},sg:56125c52971a42e2b8c90f6838ec9c1f,2019-02-25 08:00:00,163.0,0.0,2
96756297,488064,Commercial,Retail Trade,466.449554,5020.816406,41051004103,{04b67782-0d55-45cc-8a67-9e565e7c3bf1},sg:56125c52971a42e2b8c90f6838ec9c1f,2019-02-25 08:00:00,164.0,0.0,2
96756298,488064,Commercial,Retail Trade,466.449554,5020.816406,41051004103,{04b67782-0d55-45cc-8a67-9e565e7c3bf1},sg:56125c52971a42e2b8c90f6838ec9c1f,2019-02-25 08:00:00,165.0,0.0,2
96756299,488064,Commercial,Retail Trade,466.449554,5020.816406,41051004103,{04b67782-0d55-45cc-8a67-9e565e7c3bf1},sg:56125c52971a42e2b8c90f6838ec9c1f,2019-02-25 08:00:00,166.0,0.0,2
